In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv("../data/raw/superstore.csv", encoding="latin1")

df.head()
df.shape
df.columns
#df.info()
df.isnull().sum()
df.duplicated().sum()
df.describe()
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")
df[["order_date", "ship_date"]].isnull().sum()


numeric_cols = ["sales", "quantity", "discount", "profit"]
df[numeric_cols].describe()

df[df["sales"] < 0]
df[df["quantity"] <= 0]
df[df["discount"] < 0]
df[df["discount"] > 1]

df.duplicated().sum()

df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_month_name"] = df["order_date"].dt.month_name()
df["order_year_month"] = df["order_date"].dt.to_period("M").astype(str)
df["order_quarter"] = df["order_date"].dt.quarter

df[["order_date", "order_year", "order_month", "order_month_name", "order_year_month", "order_quarter"]].head()

df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["shipping_days"]

df["profit_margin"] = df["profit"] / df["sales"]
df["profit_margin"]

df["profit_margin"] = df["profit_margin"].replace([np.inf, -np.inf], np.nan)

def discount_band(x):
    if x == 0:
        return "No Discount"
    elif x <= 0.2:
        return "Low Discount"
    elif x <= 0.5:
        return "Medium Discount"
    else:
        return "High Discount"

df["discount_band"] = df["discount"].apply(discount_band)
df[["discount", "discount_band"]]

df["is_profitable"] = df["profit"].apply(lambda x: 1 if x > 0 else 0)
df[["profit", "is_profitable"]]
df.to_csv("../data/cleaned/superstore_cleaned.csv", index=False)

In [2]:
import pandas as pd

input_path = "../data/cleaned/superstore_cleaned.csv"
output_path = "../data/cleaned/superstore_cleaned_mysql.csv"

df = pd.read_csv(input_path, encoding="utf-8")

# Replace non-breaking spaces and other weird hidden spaces in text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("\u00A0", " ", regex=False)
        .str.replace("\u200B", "", regex=False)
        .str.strip()
    )

df.to_csv(output_path, index=False, encoding="utf-8")

print("Saved cleaned MySQL import file:", output_path)

Saved cleaned MySQL import file: ../data/cleaned/superstore_cleaned_mysql.csv


/var/folders/gc/skk7s1910ms6thpv4fm5xfvr0000gn/T/ipykernel_16111/1852824831.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
